# 00 — Подготовка данных GTZAN

**Что делает ноутбук:**
1. Подключает GTZAN dataset из Kaggle (`andradaolteanu/gtzan-dataset-music-genre-classification`).
2. Применяет **Sturm fault-filter** (Sturm 2014) — убирает дубликаты, mislabeled, distortions.
3. Считает log-mel-спектрограммы по PANNs-параметрам (sr=32k, n_fft=1024, hop=320, n_mels=64, fmin=50, fmax=14000).
4. Сохраняет всё в `gtzan_logmel.h5` (~500 MB, GZIP-compression).
5. Строит 5-fold stratified split + сохраняет в `folds.json`.

**Входные данные:**
- `/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/`
- `/kaggle/input/gtzan-cnn14-resnet18-src/src/` (наш код, подключить как Kaggle Dataset)

**Выходные данные:**
- `/kaggle/working/gtzan_logmel.h5`
- `/kaggle/working/folds.json`

**Время на T4:** ~15-20 мин.

In [ ]:
# Установка зависимостей (большинство уже есть в Kaggle Python image)
!pip install -q librosa==0.10.1 h5py soxr fvcore grad-cam umap-learn 2>&1 | tail -5

In [ ]:
# Подключаем наш код (загружен как Kaggle Dataset с именем 'gtzan-cnn14-resnet18-src')
import sys
from pathlib import Path

# Пробуем несколько возможных путей
candidates = [
    '/kaggle/input/gtzan-cnn14-resnet18-src',
    '/kaggle/input/cnn14-resnet18-src',
    '/kaggle/working',
    str(Path.cwd().parent),  # локальный запуск из notebooks/
]
for p in candidates:
    if Path(p, 'src', '__init__.py').exists():
        sys.path.insert(0, p)
        print(f'src/ found at: {p}')
        break
else:
    raise FileNotFoundError('Не найден src/. Загрузите его как Kaggle Dataset.')

from src import GENRES
from src.sturm_filter import get_clean_track_ids
from src.dataset import preprocess_to_hdf5, save_folds
from src.configs import MelConfig
print(f'GENRES: {GENRES}')

In [ ]:
# Конфигурация путей: автодетект между Kaggle и локалью
kaggle_gtzan = Path('/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original')
local_gtzan = Path.cwd().parent / 'GTZAN' / 'genres_original'

if kaggle_gtzan.exists():
    GENRES_DIR = kaggle_gtzan
    OUT_DIR = Path('/kaggle/working')
elif local_gtzan.exists():
    GENRES_DIR = local_gtzan
    OUT_DIR = Path.cwd().parent / 'outputs'
else:
    raise FileNotFoundError(
        'GTZAN не найден. На Kaggle подключите dataset andradaolteanu/gtzan-dataset-music-genre-classification.'
    )
OUT_DIR.mkdir(exist_ok=True, parents=True)
print(f'GTZAN: {GENRES_DIR}')
print(f'OUT:   {OUT_DIR}')

In [ ]:
# Sturm fault-filter
clean_ids = sorted(get_clean_track_ids(GENRES_DIR, method='kereliuk_or_embedded'))
print(f'Чистых треков: {len(clean_ids)}')

# Распределение по жанрам — должно быть сбалансировано после фильтрации
from collections import Counter
by_genre = Counter(tid.split('.')[0] for tid in clean_ids)
print('По жанрам:')
for g in GENRES:
    print(f'  {g:10s}: {by_genre[g]}')

In [ ]:
# Препроцессинг: WAV → log-mel → HDF5
H5_PATH = OUT_DIR / 'gtzan_logmel.h5'
preprocess_to_hdf5(
    genres_dir=GENRES_DIR,
    track_ids=clean_ids,
    out_path=H5_PATH,
    cfg=MelConfig(),
    overwrite=False,
    verbose=True,
)

In [ ]:
# Sanity check: открыть HDF5, проверить размер 1 спектрограммы
import h5py
with h5py.File(H5_PATH, 'r') as h5:
    print(f'HDF5 ключей: {len(h5.keys())}')
    print(f'Атрибуты mel: {dict(h5.attrs)}')
    sample_key = list(h5.keys())[0]
    print(f'Пример "{sample_key}": shape={h5[sample_key].shape}, dtype={h5[sample_key].dtype}, label={h5[sample_key].attrs["label"]}')

In [ ]:
# 5-fold stratified split + 10% валидация внутри train
from sklearn.model_selection import StratifiedKFold, train_test_split
import numpy as np

from src import GENRE_TO_IDX
labels = np.array([GENRE_TO_IDX[tid.split('.')[0]] for tid in clean_ids])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = {}
for fold_idx, (train_full_idx, test_idx) in enumerate(skf.split(np.zeros(len(clean_ids)), labels)):
    train_full = [clean_ids[i] for i in train_full_idx]
    test = [clean_ids[i] for i in test_idx]
    # Внутри train выделим 10% на val (stratified)
    train_full_labels = [GENRE_TO_IDX[tid.split('.')[0]] for tid in train_full]
    train, val = train_test_split(
        train_full, test_size=0.1, stratify=train_full_labels, random_state=42,
    )
    folds[str(fold_idx)] = {'train': sorted(train), 'val': sorted(val), 'test': sorted(test)}
    print(f'fold {fold_idx}: train={len(train)} val={len(val)} test={len(test)}')

FOLDS_PATH = OUT_DIR / 'folds.json'
save_folds(folds, FOLDS_PATH)
print(f'Сохранено: {FOLDS_PATH}')

In [ ]:
# Финальный sanity: визуализация одной спектрограммы
import matplotlib.pyplot as plt
import h5py

with h5py.File(H5_PATH, 'r') as h5:
    fig, axes = plt.subplots(2, 5, figsize=(20, 6))
    for i, genre in enumerate(GENRES):
        keys = [k for k in h5.keys() if k.startswith(genre)]
        if not keys:
            continue
        ax = axes[i // 5, i % 5]
        mel = h5[keys[0]][:]
        ax.imshow(mel, aspect='auto', origin='lower', cmap='magma')
        ax.set_title(f'{genre} ({keys[0]})')
        ax.set_xlabel('Time frame')
        ax.set_ylabel('Mel bin')
plt.tight_layout()
plt.savefig(OUT_DIR / 'mel_examples.png', dpi=120, bbox_inches='tight')
plt.show()

## Готово

Артефакты в `/kaggle/working/`:
- `gtzan_logmel.h5` — все log-mel спектрограммы
- `folds.json` — 5-fold splits с val
- `mel_examples.png` — по одной спектрограмме на жанр

Можно скачать `gtzan_logmel.h5` и `folds.json` как Output → подключить в ноутбуках 01/02 как Kaggle Dataset, чтобы не пересчитывать.